#### 1. Import các thư viện và cấu hình

In [5]:
import pandas as pd
import numpy as np
import warnings

In [6]:
# Cấu hình hiển thị và tắt cảnh báo dateutil
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None 

# Tham số cố định
KEEP_COLS = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 
    'total_acc', 'loan_status'
]

#### 2. Tải và lọc dữ liệu gốc

In [7]:
df = pd.read_csv('accepted_2007_to_2018Q4.csv', low_memory=False)
df_filtered = df[df.columns.intersection(KEEP_COLS)].copy()

# Xử lý Target: 1 là Approved (Nên cho vay), 0 là Rejected (Không nên cho vay)
df_filtered = df_filtered[df_filtered['loan_status'].isin(['Fully Paid', 'Charged Off'])]
df_filtered['approved'] = df_filtered['loan_status'].map({'Fully Paid': 1, 'Charged Off': 0})
df_filtered.drop(columns=['loan_status'], inplace=True)

#### 3. Feature Engineering

In [15]:
# Xử lí thời gian và biến số đơn giản
# Quy đổi thời gian tín dụng (Credit Age) tính đến mốc 2018-12
df_filtered['earliest_cr_line'] = pd.to_datetime(df_filtered['earliest_cr_line'])
df_filtered['credit_hist_months'] = (pd.to_datetime('2018-12-01') - df_filtered['earliest_cr_line']).dt.days // 30

# Chuyển đổi định dạng số cho emp_length và term
df_filtered['emp_length'] = df_filtered['emp_length'].astype(str).str.extract('(\d+)').astype(float).fillna(0)
df_filtered['term_num'] = df_filtered['term'].astype(str).str.extract('(\d+)').astype(float)

# Tỷ lệ trả nợ trên thu nhập (Biến "vàng")
df_filtered['inst_to_inc_ratio'] = df_filtered['installment'] / (df_filtered['annual_inc'] / 12 + 1)

In [16]:
# Xử lý Outlier và Log Transform cho các biến bị lệch (Skewed)
upper_inc = df_filtered['annual_inc'].quantile(0.95)
df_filtered['annual_inc'] = df_filtered['annual_inc'].clip(upper=upper_inc)

for col in ['annual_inc', 'revol_bal', 'loan_amnt']:
    df_filtered[f'log_{col}'] = np.log1p(df_filtered[col])

# Target Encoding cho địa lý (State Risk Score)
state_risk = df_filtered.groupby('addr_state')['approved'].mean()
df_filtered['state_risk_score'] = df_filtered['addr_state'].map(state_risk)
# Gom nhóm mục đích vay (Purpose Grouping)
purpose_groups = {
    'credit_card': 'debt', 'debt_consolidation': 'debt',
    'home_improvement': 'investment', 'small_business': 'investment',
    'major_purchase': 'expense', 'medical': 'expense', 'wedding': 'expense'
}
df_filtered['purpose_grouped'] = df_filtered['purpose'].map(purpose_groups).fillna('other')

In [17]:
# Danh sách features tinh gọn cho mô hình XAI/DNN
FINAL_FEATURES = [
    'log_loan_amnt', 'term_num', 'sub_grade', 'emp_length', 'home_ownership', 
    'log_annual_inc', 'purpose_grouped', 'dti', 'revol_util', 'log_revol_bal',
    'credit_hist_months', 'inst_to_inc_ratio', 'state_risk_score', 'approved'
]

#### 4. Lấy sample

In [18]:
# Lấy mẫu 10,000 dòng 
df_10k = pd.concat([
    df_filtered[df_filtered['approved'] == 1].sample(5000, random_state=42),
    df_filtered[df_filtered['approved'] == 0].sample(5000, random_state=42)
]).sample(frac=1, random_state=42) # Xáo trộn lại dữ liệu

# Xuất file phục vụ huấn luyện
df_10k[FINAL_FEATURES].to_csv('lendingclub.csv', index=False)
print("Đã xuất file lendingclub.csv thành công với 10,000 bản ghi.")

Đã xuất file lendingclub.csv thành công với 10,000 bản ghi.
